In [1]:
import requests
import pandas as pd
import time

# --- 1. Core keyword list (140+ targets) ---
planets = [
    "Mercury", "Venus", "Earth", "Mars", "Jupiter", "Saturn", "Uranus", "Neptune", "Pluto", "Ceres", 
    "Eris", "Haumea", "Makemake", "Proxima b", "TRAPPIST-1e", "Kepler-186f", "Kepler-22b", "WASP-12b", 
    "55 Cancri e", "HD 189733b", "Gliese 581g", "Kepler-452b", "TOI-700 d", "LHS 1140 b", 
    "Kepler-16b", "HD 209458b", "Kepler-10b", "51 Pegasi b", "Ross 128 b", "Wolf 1061c",
    "Gliese 667 Cc", "Kepler-442b", "Kepler-62f", "Kepler-1229b", "K2-18b", "HD 131399 Ab", 
    "WASP-76b", "HAT-P-7b", "Kepler-7b", "Kepler-13b"
]

stars = [
    "Sun", "Sirius", "Alpha Centauri", "Betelgeuse", "Vega", "Rigel", "Arcturus",
    "Canopus", "Capella", "Procyon", "Achernar", "Altair", "Aldebaran", "Antares",
    "Spica", "Pollux", "Fomalhaut", "Deneb", "Regulus", "Castor", "Bellatrix",
    "Alnilam", "Alnitak", "Saiph", "Algol", "Polaris", "Eta Carinae", "Mira",
    "Denebola", "Alcor", "Mizar", "Thuban", "Kochab", "Alkaid", "Dubhe",
    "Regor", "Avior", "Shaula", "Sargas", "Peacock"
]

nebulae = [
    "Orion Nebula", "Eagle Nebula", "Carina Nebula", "Crab Nebula", "Helix Nebula", 
    "Ring Nebula", "Veil Nebula", "Lagoon Nebula", "Trifid Nebula", "Rosette Nebula", 
    "Horsehead Nebula", "Cat's Eye Nebula", "Butterfly Nebula", "Eskimo Nebula", 
    "North America Nebula", "Tarantula Nebula", "Dumbbell Nebula", "Bubble Nebula", 
    "Medusa Nebula", "Pillars of Creation", "Heart Nebula", "Soul Nebula",
    "Wizard Nebula", "Elephant's Trunk Nebula", "California Nebula", "Iris Nebula", 
    "Pacman Nebula", "Crescent Nebula", "Pelican Nebula", "Jellyfish Nebula", 
    "Cone Nebula", "Flame Nebula", "Ghost Nebula", "Skull Nebula", "Thor's Helmet", 
    "Lobster Nebula", "Statue of Liberty Nebula", "Running Chicken Nebula", 
    "Prawn Nebula", "Monkey Head Nebula", "Little Ghost Nebula", "Saturn Nebula", 
    "Owl Nebula", "Spider Nebula", "Flaming Star Nebula", "Cave Nebula", "Godzilla Nebula",
    "Shark Nebula", "Tulip Nebula", "Wolf Cave Nebula", "Blinking Planetary Nebula",
    "Blue Snowball Nebula", "Bowtie Nebula", "Bug Nebula", "Christmas Tree Nebula", "Cooling Tower Nebula",
    "Cotton Candy Nebula", "Egg Nebula", "Eight-Burst Nebula", "Finger of God Nebula", "Fried Egg Nebula"
]

queries_dict = {"Planet": planets, "Star": stars, "Nebula": nebulae}
NASA_API_URL = "https://images-api.nasa.gov/search"

# --- 2. Clean filter: restrict trusted centers and remove noisy images ---
def is_valid_science_image(info):
    center = info.get('center', '')
    title = info.get('title', '').lower()
    desc = info.get('description', '').lower()
    
    # A. Core strategy: allow only JPL and STScI (STScI images are often marked as centers or mentioned in the description)
    # Note: STScI images may sometimes be tagged as GSFC or JPL, but 'STScI' or 'Webb/Hubble' will appear in title/description.
    allowed_centers = ['JPL', 'STScI']
    
    # If the center is not trusted and the description does not mention Hubble/Webb, it is likely not a deep space science image.
    is_trusted_source = center in allowed_centers or any(x in (title + desc) for x in ['hubble', 'webb', 'stsci', 'spitzer'])
    
    if not is_trusted_source:
        return False

    # B. Blacklist filter: remove non-scientific images (people, models, facilities, etc.) even from trusted centers.
    blacklist = ['staff', 'people', 'technician', 'meeting', 'presentation', 'facility', 'model', 'exhibit', 'museum', 'standing', 'posing']
    if any(word in (title + desc) for word in blacklist):
        return False
        
    return True

# --- 3. Scraping logic ---
def fetch_nasa_science_data(queries_dict):
    all_records = []
    for category, queries in queries_dict.items():
        for q in queries:
            print(f"📡 Deep scan [{category}]: {q} ...")
            # Because we lock to trusted centers, quality is high and we can iterate more pages to increase coverage.
            for page in range(1, 4): 
                params = {'q': q, 'media_type': 'image', 'page': page}
                try:
                    resp = requests.get(NASA_API_URL, params=params, timeout=15)
                    items = resp.json().get('collection', {}).get('items', [])
                    if not items: break
                    
                    for item in items:
                        info = item['data'][0]
                        if is_valid_science_image(info):
                            all_records.append({
                                "category": category,
                                "query_term": q,
                                "nasa_id": info.get('nasa_id'),
                                "title": info.get('title'),
                                "description": info.get('description'),
                                "center": info.get('center'),
                                "img_url": item['links'][0]['href'] if 'links' in item else None,
                                "date": info.get('date_created')
                            })
                    time.sleep(0.1)
                except: break
            print(f"   📊 Valid records collected so far: {len(all_records)}")
            
    return all_records

# --- 4. Run, deduplicate, and persist dataset ---
raw_results = fetch_nasa_science_data(queries_dict)
df = pd.DataFrame(raw_results)

if not df.empty:
    df = df.drop_duplicates(subset=['nasa_id'])

    df.to_csv("nasa_deep_space_jpl_stsci.csv", index=False, encoding='utf-8-sig')
    print(f"\nClean dataset construction complete!")
    print(f"Final scale: {len(df)} records (JPL & STScI Core Libraries)")
else:
    print("❌ No qualifying deep-space data was captured.")

📡 Deep scan [Planet]: Mercury ...
   📊 Valid records collected so far: 244
📡 Deep scan [Planet]: Venus ...
   📊 Valid records collected so far: 420
📡 Deep scan [Planet]: Earth ...
   📊 Valid records collected so far: 484
📡 Deep scan [Planet]: Mars ...
   📊 Valid records collected so far: 734
📡 Deep scan [Planet]: Jupiter ...
   📊 Valid records collected so far: 1021
📡 Deep scan [Planet]: Saturn ...
   📊 Valid records collected so far: 1253
📡 Deep scan [Planet]: Uranus ...
   📊 Valid records collected so far: 1319
📡 Deep scan [Planet]: Neptune ...
   📊 Valid records collected so far: 1440
📡 Deep scan [Planet]: Pluto ...
   📊 Valid records collected so far: 1600
📡 Deep scan [Planet]: Ceres ...
   📊 Valid records collected so far: 1886
📡 Deep scan [Planet]: Eris ...
   📊 Valid records collected so far: 1890
📡 Deep scan [Planet]: Haumea ...
   📊 Valid records collected so far: 1890
📡 Deep scan [Planet]: Makemake ...
   📊 Valid records collected so far: 1890
📡 Deep scan [Planet]: Proxima b 